In [ ]:
from os import add_dll_directory
!pip install numpy pandas plotly

In [1]:
!pip install folium

/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
  Using cached folium-0.20.0-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached xyzservices-2025.4.0-py3-none-any.whl.metadata (4.3 kB)
Using cached folium-0.20.0-py2.py3-none-any.whl (113 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 11.2 MB/s eta 0:00:0000:0100:01
Using cached xyzservices-2025.4.0-py3-none-any.whl (90 kB)

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
!pip install scikit-learn

In [ ]:
!pip install geopandas

In [3]:
import folium
from folium.plugins import MarkerCluster
import calendar
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import learning_curve
import numpy as np
from sklearn.model_selection import train_test_split, KFold,GridSearchCV, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error
import os
import pandas as pd
import plotly.express as px
from scipy.stats import randint

In [4]:
# Fájl elérési útja
csv_path = 'data/crime_data.csv'

# Beolvasás
df = pd.read_csv(csv_path)

# Első pár sor
df.head()

,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,13974502,JJ423750,09/21/2025 12:00:00 AM,095XX S CALUMET AVE,1310,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE - GARAGE,False,False,...,9.0,49.0,14,1180262.0,1841791.0,2025,09/28/2025 03:42:59 PM,41.721137,-87.615333,"(41.72113678, -87.615333015)"
1,13975969,JJ425352,09/21/2025 12:00:00 AM,031XX S WELLS ST,0710,THEFT,THEFT FROM MOTOR VEHICLE,RESIDENCE,False,False,...,11.0,34.0,06,1175131.0,1884127.0,2025,09/28/2025 03:42:59 PM,41.837427,-87.632864,"(41.837427273, -87.632863567)"
2,13974043,JJ423022,09/21/2025 12:00:00 AM,001XX W LAKE ST,0810,THEFT,OVER $500,SIDEWALK,False,False,...,42.0,32.0,06,1175401.0,1901736.0,2025,09/28/2025 03:42:59 PM,41.885742,-87.631345,"(41.885741529, -87.631344503)"
3,13976000,JJ425621,09/21/2025 12:00:00 AM,050XX N WOLCOTT AVE,1153,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,RESIDENCE,False,False,...,40.0,4.0,11,1162901.0,1933808.0,2025,09/28/2025 03:42:59 PM,41.974020,-87.676344,"(41.974020487, -87.676344418)"
4,13976430,JJ425187,09/21/2025 12:00:00 AM,021XX S CALUMET AVE,1320,CRIMINAL DAMAGE,TO VEHICLE,HOTEL / MOTEL,False,False,...,3.0,33.0,14,1178840.0,1890202.0,2025,09/28/2025 03:42:59 PM,41.854014,-87.619068,"(41.854013684, -87.619068382)"


In [ ]:
import sqlite3

# Létrehozunk egy SQLite adatbázist memóriában
conn = sqlite3.connect(":memory:")

# CSV-t importáljuk táblaként
df.to_sql("crime", conn, index=False, if_exists='replace')



In [ ]:
def treemap(categories,title,path,values):
    fig = px.treemap(categories, path=path, values=values, height=700,
                 title=title, color_discrete_sequence = px.colors.sequential.RdBu)
    fig.data[0].textinfo = 'label+text+value'
    fig.show()



In [ ]:
def histogram(data,path,color,title,xaxis,yaxis):
    fig = px.histogram(data, x=path,color=color)
    fig.update_layout(
        title_text=title,
        xaxis_title_text=xaxis,
        yaxis_title_text=yaxis,
        bargap=0.2,
        bargroupgap=0.1
    )
    fig.show()

In [ ]:
Number_crimes = df['Primary Type'].value_counts()

categories = pd.DataFrame({
    'Primary Type': Number_crimes.index,
    'values': Number_crimes.values
})
categories['values'] = pd.to_numeric(categories['values'], errors='coerce')
print(categories)
print(categories.head())

In [ ]:
treemap(categories,'Major Crimes in Chicago',['Primary Type'],categories['values'])

In [ ]:
# Create a map visualization of the data
m = folium.Map(location=[df['Latitude'].mean(), df['Longitude'].mean()],zoom_start=12)
#Visualize the theft crimes from df on the map
theft = df[df['Primary Type'] == 'INTERFERENCE WITH PUBLIC OFFICER']
valid_theft = theft.dropna(subset=['Latitude', 'Longitude'])
marker_cluster = MarkerCluster().add_to(m)
for _, row in valid_theft.iterrows():  # Use valid_theft instead of theft
    folium.Marker(
       location=[row['Latitude'], row['Longitude']],
       popup=str(row['Date'])
    ).add_to(marker_cluster)
m


## Főbb bűncselekmények megjelenítése kerületenként

In [ ]:
district_crimes = df.groupby(['District', 'Primary Type']).size().reset_index(name='count')
district_crimes

In [ ]:
fig = px.bar(district_crimes,
             x='District',
             y='count',
             color='Primary Type',
             title='Főbb bűncselekmények eloszlása kerületenként',
             labels={'District': 'Kerület', 'count': 'Bűncselekmények száma',
                     'Primary Type': 'Bűncselekmény típusa'},
             height=600)
fig.update_layout(xaxis_type='category')
fig.show()

In [ ]:
top_crimes = categories.head(10)['Primary Type'].tolist()
district_top_crimes = df[df['Primary Type'].isin(top_crimes)].groupby(
    ['District', 'Primary Type']).size().reset_index(name='count')

fig = px.sunburst(district_top_crimes,
                  path=['District', 'Primary Type'],
                  values='count',
                  title='Top 10 bűncselekmény kerületenként (Sunburst diagram)',
                  height=700)
fig.show()


# Predict the daily number of crimes per month in 2025 using the using data from 2001-2024

## 1. Prepare data

In [1]:
#create new df
data_reduced = df[['ID','District','Primary Type','Date', 'Latitude', 'Longitude']].copy()
#Split the date to determine year and month
data_reduced['Year'] = df['Date'].str.split("/", expand=True)[2].str.split(" ", expand=True)[0]
data_reduced['Month'] = df['Date'].str.split("/", expand=True)[0]



NameError: name 'df' is not defined

In [ ]:
#Count how many crime happened in each month in each year in each district
Number_crimes_sum = data_reduced.groupby(['District','Year','Month']).size().reset_index(name='count')
#Calculate the number of crimes per day in the month
Number_crimes_sum['Number_of_days_in_month'] = Number_crimes_sum.apply(
    lambda row: calendar.monthrange(int(row['Year']), int(row['Month']))[1],
    axis=1)

Number_crimes_sum['Number of Crimes Per Day'] = Number_crimes_sum['count'] / Number_crimes_sum['Number_of_days_in_month']
Number_crimes_sum['Year'] = pd.to_numeric(Number_crimes_sum['Year'])


In [ ]:
# The training data will be 2001-2024 and we will predict to 2025
Traning_data = Number_crimes_sum[Number_crimes_sum['Year'].between(2001, 2024)]
Traning_data =Traning_data.drop(['Number_of_days_in_month', 'count'], axis=1)
X = Traning_data.drop(['Number of Crimes Per Day'], axis=1).copy()
Y = Traning_data['Number of Crimes Per Day'].copy()

In [ ]:
#We will predict to 2025
Test_data = Number_crimes_sum[(Number_crimes_sum['Year'] == 2025)]
Test_data = Test_data.drop(['Number_of_days_in_month', 'count'], axis=1)
Y_check = Test_data['Number of Crimes Per Day']
Test_data = Test_data.drop(['Number of Crimes Per Day'], axis=1)

## 2. Random Forest

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
# Use Random Forest model to predict
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train,y_train)
y_pred = model.predict(X_train)
print(model.score(X_train,y_train))



In [ ]:
# Calculate RMSE, MAPE, Accuracy of the model
predictions = model.predict(X_train)
MSE = mean_squared_error(y_train, predictions)
RMSE = np.sqrt(MSE)
errors = abs(predictions - y_train)
MAPE = np.mean(100 * (errors / y_train))
Accuracy = 100 - MAPE
print('RMSE of the model:', RMSE)
print('MAPE of the model:', MAPE)
print('Accuracy of the model:', Accuracy, '%')

In [ ]:
sns.scatterplot(x=y_train, y=y_pred)
plt.xlabel("Valós értékek")
plt.ylabel("Prediktált értékek")
plt.title("Valós vs. Prediktált értékek - Random Forest Regresszor")
plt.show()


train_sizes, train_scores, test_scores = learning_curve(model, X_train, y_train, cv=5)

plt.plot(train_sizes, np.mean(train_scores, axis=1), label="Train score")
plt.plot(train_sizes, np.mean(test_scores, axis=1), label="Test score")
plt.legend()
plt.title("Learning curve - Random Forest")
plt.xlabel("Train minták száma")
plt.ylabel("Pontosság / R²")
plt.show()

### 2.1. Hyperparameter tuning

In [ ]:
#Hyperparameter tuning using GridSearchCV if the files are not exist
if not os.path.exists('parameters/best_param.txt') or not os.path.exists('parameters/best_estimator.txt'):
    from sklearn.model_selection import GridSearchCV
    param = {
        'n_estimators': [10, 25, 80, 100, 1000],
        'max_features': [5, 10],
        'max_depth': [None, 10, 80, 90, 100],
        'bootstrap': [True, False]
    }
    grid_search = GridSearchCV(param_grid=param, estimator=model, cv=10, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)

    print(grid_search.best_params_)
    print(grid_search.best_estimator_)
    best_param = grid_search.best_params_
    best_estimator = grid_search.best_estimator_
    # Save the best_param to a txt file and save the txt file to the same folder as the notebook
    with open('parameters/best_param.txt', 'w') as f:
        f.write(str(best_param))
        f.close()
    # Save the best_estimator to a txt file and save the txt file to the same folder as the notebook
    with open('parameters/best_estimator.txt', 'w') as f:
        f.write(str(best_estimator))
        f.close()
else:
    #Read the best_param file
    with open('parameters/best_param.txt', 'r') as f:
        best_param = eval(f.read())
        f.close()

    #Read the best_estimator file
    with open('parameters/best_estimator.txt', 'r') as f:
        best_estimator = eval(f.read())
        f.close()

In [ ]:
#Calculate the accuracy and RMSE of the model
best_estimator.fit(X_train, y_train)
y_pred_gridbest = best_estimator.predict(X_train)
#y_pred_gridbest = grid_search.best_estimator_.predict(X_train)
#RMSE
MSE = mean_squared_error(y_train, y_pred_gridbest)
RMSE = np.sqrt(MSE)
print('RMSE of the model:', RMSE)
#Accuracy
errors = abs(y_pred_gridbest - y_train)
MAPE = np.mean(100 * (errors / y_train))
Accuracy = 100 - MAPE
print('Accuracy of the model:', Accuracy, '%')

In [ ]:
#Hyperparameter tuning using RandomizedSearchCV
if not os.path.exists('parameters/best_param_random.txt') or not os.path.exists('parameters/best_estimator_random.txt'):
    param_random = {
        'n_estimators': [int(x) for x in np.linspace(start = 10, stop = 200, num = 5)],
        'max_features': range(1, 11),
        'max_depth': [int(x) for x in np.linspace(1, 100, num = 3)],
        'bootstrap': [True, False]
    }
    random_search = RandomizedSearchCV(model, param_random, n_iter=10, cv=10, verbose=1, random_state=42, n_jobs=-1, scoring='neg_mean_squared_error')

    random_search.fit(X_train, y_train)
    print(random_search.best_params_)
    print(random_search.best_estimator_)
    best_param_random = random_search.best_params_
    best_estimator_random = random_search.best_estimator_
    # Save the best_param to a txt file and save the txt file to the same folder as the notebook
    with open('parameters/best_param_random.txt', 'w') as f:
        f.write(str(best_param_random))
        f.close()
    # Save the best_estimator to a txt file and save the txt file to the same folder as the notebook
    with open('parameters/best_estimator_random.txt', 'w') as f:
        f.write(str(best_estimator_random))
        f.close()
else:
    #Read the best_param file
    with open('parameters/best_param_random.txt', 'r') as f:
        best_param_random = eval(f.read())
        f.close()

    #Read the best_estimator file
    with open('parameters/best_estimator_random.txt', 'r') as f:
        best_estimator_random = eval(f.read())
        f.close()

In [ ]:
#Calculate the accuracy and RMSE of the model
best_estimator_random.fit(X_train, y_train)
y_pred_randombest = best_estimator_random.predict(X_train)
#y_pred_randombest = random_search.best_estimator_.predict(X_train)
#RMSE
MSE = mean_squared_error(y_train, y_pred_randombest)
RMSE = np.sqrt(MSE)
print('RMSE of the model:', RMSE)
#Accuracy
errors = abs(y_pred_randombest - y_train)
MAPE = np.mean(100 * (errors / y_train))
Accuracy = 100 - MAPE
print('Accuracy of the model:', Accuracy, '%')

### 2.2 Feature importance

In [ ]:
#Feature Importance
feature_importance = pd.DataFrame({})
feature_importance['feature'] = X_train.columns
feature_importance['importance'] = best_estimator.feature_importances_
#Visualize the feature importance
plt.figure(figsize=(10, 6))
plt.bar(feature_importance['feature'], feature_importance['importance'])
plt.title('Feature Importance - Random Forest')


### 2.3 Final Model

In [ ]:
#using the best parameters on test data
final_model = RandomForestRegressor(**best_param, random_state=42)
final_model.fit(X_train, y_train)
y_pred_final = final_model.predict(X_test)

#RMSE
MSE = mean_squared_error(y_test, y_pred_final)
RMSE = np.sqrt(MSE)
print('RMSE of the model:', RMSE)
#Accuracy
errors = abs(y_pred_final - y_test)
MAPE = np.mean(100 * (errors / y_test))
Accuracy = 100 - MAPE
print('Accuracy of the model:', Accuracy, '%')


### 2.4 Use the final model to predict data for 2025

In [ ]:
#Use on 2025 data
y_pred_check = final_model.predict(Test_data)
#RMSE
MSE = mean_squared_error(Y_check, y_pred_check)
RMSE = np.sqrt(MSE)
print('RMSE of the model:', RMSE)
#Accuracy
errors = abs(y_pred_check - Y_check)
MAPE = np.mean(100 * (errors / Y_check))
Accuracy = 100 - MAPE
print('Accuracy of the model:', Accuracy, '%')
#Visualize the results of the prediction
plt.scatter(y_pred_check, Y_check)



## 3. Add new columns to data

In [ ]:
#Copy the data
X_extended = X.copy()
Test_data_extended = Test_data.copy()
for df in (X_extended, Test_data_extended):
    #Month to float
    df['Month'] = df['Month'].astype(float)
    # Add interactions
    df['District_Year_Interaction'] = df['District'] * df['Year']
    # Month sin
    df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12)
    #df['Quarter'] = ((df['Month'] - 1) // 3) + 1
    df['Is_Winter'] = df['Month'].isin([12, 1, 2]).astype(int)
    df['Is_Summer'] = df['Month'].isin([6, 7, 8]).astype(int)
    #Drop the month
    df.drop('Month', axis=1, inplace=True)



### 3.0 Hyperparameter tuning with the new model TO DO

### 3.1 Use the final model on traning data


In [ ]:
# Split the data into training and testing sets
X_train_ex, X_test_ex, y_train_ex, y_test_ex = train_test_split(X_extended, Y, test_size=0.2, random_state=42)
# Train the final model
final_model.fit(X_train_ex, y_train_ex)

In [ ]:
y_pred_final_ex = final_model.predict(X_test_ex)
#RMSE
MSE = mean_squared_error(y_test_ex, y_pred_final_ex)
RMSE = np.sqrt(MSE)
print('RMSE of the model:', RMSE)
#Accuracy
errors = abs(y_pred_final_ex - y_test_ex)
MAPE = np.mean(100 * (errors / y_test_ex))
Accuracy = 100 - MAPE
print('Accuracy of the model:', Accuracy, '%')

### 3.2 Use the final model on the extended test data

In [ ]:
# Use the final model on the extended test data
y_pred_check_ex = final_model.predict(Test_data_extended)
#RMSE
MSE = mean_squared_error(Y_check, y_pred_check_ex)
RMSE = np.sqrt(MSE)
print('RMSE of the model:', RMSE)
#Accuracy
errors = abs(y_pred_check_ex - Y_check)
MAPE = np.mean(100 * (errors / Y_check))
Accuracy = 100 - MAPE
print('Accuracy of the model:', Accuracy, '%')

## 4. Visualize the results of the prediction on a map

In [ ]:
import geopandas as gpd

# Import geojson file for districts
districts_gdf = gpd.read_file("data/PoliceDistrictDec2012_20251008.geojson")
districts_gdf = districts_gdf.drop([':updated_at', ':created_at'], axis=1)

### 4.1 Add the predicted values to the data

In [ ]:
Test_data_with_pred = Test_data.copy()
Test_data_with_pred['Predicted_Crimes_Per_Day'] = y_pred_check
Test_data_with_pred['Actual_Crimes_Per_Day'] = Y_check.values
Test_data_with_pred

### 4.2 Add longitude and latitude to each district

In [ ]:
# Add the mean of the predicted and actual values for each district
pred_by_district = Test_data_with_pred.groupby('District').agg({
    'Predicted_Crimes_Per_Day': 'mean',
    'Actual_Crimes_Per_Day': 'mean'
}).reset_index()
pred_by_district['District'] = pred_by_district['District'].astype(int)
districts_gdf['dist_num'] = districts_gdf['dist_num'].astype(int)

# Merge the predicted and actual values with the districts data
districts_with_data = pd.merge(
    districts_gdf,
    pred_by_district,
    left_on='dist_num',
    right_on='District',
    how='inner')

### 4.3 Visualize the results of the prediction on a map

In [ ]:
m_choropleth_pred = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

folium.Choropleth(
    geo_data=districts_with_data,
    name='Prediktált bűncselekmények',
    data=districts_with_data,
    columns=['dist_num', 'Predicted_Crimes_Per_Day'],
    key_on='feature.properties.dist_num',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Átlagos prediktált bűncselekmények naponta',
    nan_fill_color='lightgray'
).add_to(m_choropleth_pred)

folium.GeoJson(
    districts_with_data,
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'black', 'weight': 2},
    tooltip=folium.GeoJsonTooltip(
        fields=['dist_num', 'Predicted_Crimes_Per_Day', 'Actual_Crimes_Per_Day'],
        aliases=['Kerület:', 'Átlag prediktált:', 'Átlag valós:'],
        localize=True
    )
).add_to(m_choropleth_pred)

m_choropleth_pred

### 4.4 Visualize the actual value on the map

In [ ]:
m_choropleth_actual = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

folium.Choropleth(
    geo_data=districts_with_data,
    name='Valós bűncselekmények',
    data=districts_with_data,
    columns=['dist_num', 'Actual_Crimes_Per_Day'],
    key_on='feature.properties.dist_num',
    fill_color='YlGnBu',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Átlagos valós bűncselekmények naponta',
    nan_fill_color='lightgray'
).add_to(m_choropleth_actual)

folium.GeoJson(
    districts_with_data,
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'black', 'weight': 2},
    tooltip=folium.GeoJsonTooltip(
        fields=['dist_num', 'Predicted_Crimes_Per_Day', 'Actual_Crimes_Per_Day'],
        aliases=['Kerület:', 'Átlag prediktált:', 'Átlag valós:'],
        localize=True
    )
).add_to(m_choropleth_actual)

m_choropleth_actual

### 4.5 Visualize the difference on the map

In [ ]:
districts_with_data['Difference'] = abs(
    districts_with_data['Predicted_Crimes_Per_Day'] -
    districts_with_data['Actual_Crimes_Per_Day']
)

m_choropleth_diff = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

folium.Choropleth(
    geo_data=districts_with_data,
    name='Predikció pontossága',
    data=districts_with_data,
    columns=['dist_num', 'Difference'],
    key_on='feature.properties.dist_num',
    fill_color='RdYlGn_r',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Átlagos eltérés (Prediktált - Valós)',
    nan_fill_color='lightgray'
).add_to(m_choropleth_diff)

folium.GeoJson(
    districts_with_data,
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'black', 'weight': 2},
    tooltip=folium.GeoJsonTooltip(
        fields=['dist_num', 'Predicted_Crimes_Per_Day', 'Actual_Crimes_Per_Day', 'Difference'],
        aliases=['Kerület:', 'Átlag prediktált:', 'Átlag valós:', 'Átlagos eltérés:'],
        localize=True
    )
).add_to(m_choropleth_diff)

m_choropleth_diff
